# Multi-dataset PFN runs from data/ folder (bootstrap import)

This variant bootstraps `sys.path` to import the local `med3pipe` package without installing.

In [5]:
# Keep your existing env caps
import os, sys, site, torch
os.environ['PYTHONNOUSERSITE'] = '1'
usr = site.getusersitepackages(); sys.path = [p for p in sys.path if p != usr]

# Cap threads broadly
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

# If you still see OpenMP warnings/hangs, you can try (may slow MKL but stabilizes):
# os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'

# Optional: allow duplicate OpenMP (stops hard-crash; you already set this)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Also cap PyTorch intra-op threads
torch.set_num_threads(1)
print("Torch threads set to 1")

Torch threads set to 1


In [3]:
import sys
from pathlib import Path

def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / 'med3pipe').is_dir():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            print('Added repo root to sys.path:', base)
            return base
    raise RuntimeError("Could not locate 'med3pipe/' in current or parent directories.")

_add_repo_root_to_sys_path()


Added repo root to sys.path: C:\Users\cahel\Desktop\Med3Tab-PFN


WindowsPath('C:/Users/cahel/Desktop/Med3Tab-PFN')

In [ ]:
from med3pipe.pipelines import run_pipeline

res = run_pipeline(
    config_path="configs/datasets.yaml",
    methods=("tabpfn", "localpfn"),
    outputs_base_dir="notebooks",
)

Now import and run the multi-dataset folder-driven pipeline.

In [ ]:
# Print where the summary CSV is written and preview a few rows
summary_path = res.get('summary_path')
print('Summary CSV:', summary_path)
try:
    import pandas as pd
    df = res['summary_df']
    display(df.head(20))
except Exception as e:
    print('Could not display summary:', e)
